# Providers 03 - vLLM (CLI)

Notebook CLI paralelo a `tutorials/providers/03_vllm.ipynb`.

**Objetivo:** Inspeccionar readiness y los cuatro cruces Framework de vLLM.

Este notebook no llama factories de Agentic Systems directamente: ejecuta el
entrypoint CLI real, conserva la salida Rich y valida después el JSON del mismo
contrato.


## Cómo se ejecuta

La forma portable es `python -m agentic_systems.cli ...`. Después de instalar
el wheel, el entrypoint equivalente es `agentic-systems ...`.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def _repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio.")


ROOT = _repo_root()
CLI = [sys.executable, "-m", "agentic_systems.cli"]


def run_cli(*args: str, expected: int = 0) -> str:
    env = os.environ.copy()
    source_path = str(ROOT / "src")
    env["PYTHONPATH"] = (
        source_path
        if not env.get("PYTHONPATH")
        else source_path + os.pathsep + env["PYTHONPATH"]
    )
    command = [*CLI, *args]
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )
    print("$ " + " ".join(command))
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    assert completed.returncode == expected, completed.stderr
    return completed.stdout


def run_cli_json(*args: str) -> dict:
    return json.loads(run_cli(*args, "--json"))


def assert_rich(output: str, title: str) -> None:
    assert title in output
    assert "+" in output and "|" in output


## 1) Salida humana Rich

La celda conserva stdout y comprueba título y bordes. Esto detecta tablas o
paneles truncados, además del exit code.


In [2]:
rich_output = run_cli(*['doctor'])
assert_rich(rich_output, 'Agentic Systems Doctor')


$ C:\Python314\python.exe -m agentic_systems.cli doctor
+-------------------------- Agentic Systems Doctor ---------------------------+
| Agentic Systems 2.0.0                                                       |
| Python: 3.14.2                                                              |
| Engines: bedrock-runtime, openai-runtime, ollama-runtime, python-runtime,   |
| vllm-runtime                                                                |
| .env loaded: True                                                           |
+-----------------------------------------------------------------------------+
       Environment                         Optional Dependencies               
 VLLM_BASE_URL    missing                boto3           available             
 OPENAI_API_KEY   set                    langgraph       available             
 OLLAMA_BASE_URL  set                    openai          available             
 OLLAMA_MODEL     set                    openai-agents   availab

## 2) Contrato de máquina

La misma ruta se ejecuta con `--json` para afirmar campos y cardinalidad sin
parsear la presentación Rich.


In [3]:
args = ['matrix', 'check', '--provider', 'vllm-runtime']
live_flag = os.getenv("RUN_CLI_LIVE", "0").strip().lower() in {"1", "true", "yes"}
if live_flag:
    args.append("--live")
    args.append("--require-pass")
payload = run_cli_json(*args)

assert payload["combination_count"] == 4
assert payload["failed"] == 0
assert len(payload["results"]) == 4
if live_flag:
    assert payload["passed"] == 4
payload


$ C:\Python314\python.exe -m agentic_systems.cli matrix check --provider vllm-runtime --json
{
  "combination_count": 4,
  "failed": 0,
  "framework_filter": null,
  "live": false,
  "not_run": 4,
  "passed": 0,
  "provider_filter": "vllm-runtime",
  "results": [
    {
      "execution": "not-run",
      "execution_reason": "Pass --live to cross an external provider boundary.",
      "framework": "native",
      "offline_certified": true,
      "provider": "vllm-runtime",
      "ready": false,
      "reason": "Configure vllm-runtime before live execution.",
      "status": "needs-configuration"
    },
    {
      "execution": "not-run",
      "execution_reason": "Pass --live to cross an external provider boundary.",
      "framework": "langgraph",
      "offline_certified": true,
      "provider": "vllm-runtime",
      "ready": false,
      "reason": "Configure vllm-runtime before live execution.",
      "status": "needs-configuration"
    },
    {
      "execution": "not-run",
      "

{'combination_count': 4,
 'failed': 0,
 'framework_filter': None,
 'live': False,
 'not_run': 4,
 'passed': 0,
 'provider_filter': 'vllm-runtime',
 'results': [{'execution': 'not-run',
   'execution_reason': 'Pass --live to cross an external provider boundary.',
   'framework': 'native',
   'offline_certified': True,
   'provider': 'vllm-runtime',
   'ready': False,
   'reason': 'Configure vllm-runtime before live execution.',
   'status': 'needs-configuration'},
  {'execution': 'not-run',
   'execution_reason': 'Pass --live to cross an external provider boundary.',
   'framework': 'langgraph',
   'offline_certified': True,
   'provider': 'vllm-runtime',
   'ready': False,
   'reason': 'Configure vllm-runtime before live execution.',
   'status': 'needs-configuration'},
  {'execution': 'not-run',
   'execution_reason': 'Pass --live to cross an external provider boundary.',
   'framework': 'openai-agents',
   'offline_certified': True,
   'provider': 'vllm-runtime',
   'ready': False,
 

## Resultado e interpretación

Rich responde a lectura humana; JSON responde a automatización. Ambos nacen del
mismo comando y del mismo escenario público. Un estado `not-run` conserva el
motivo, pero no cuenta como evidencia live.
